# KUL CV GA2 — Training Notebook (Kaggle GPU)
Self-contained: no local file imports. Run all cells top-to-bottom.
After training, download `/kaggle/working/best_model.pth` and `final_model.pth`.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
DATA_DIR   = Path('/kaggle/input/competitions/kul-computer-vision-ga-2-2026')
OUTPUT_DIR = Path('/kaggle/working')

LABELS = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor',
]

BACKBONE      = 'efficientnet_b3'
IMG_SIZE      = 320
BATCH_SIZE    = 32   # T4/P100 can handle 32 at 320×320
NUM_WORKERS   = 4
VAL_SPLIT     = 0.2
RANDOM_SEED   = 42
WEIGHT_DECAY  = 1e-4

STAGE1_EPOCHS, STAGE1_LR = 5,  1e-3
STAGE2_EPOCHS, STAGE2_LR = 20, 1e-4
STAGE3_EPOCHS, STAGE3_LR = 5,  5e-5

In [ ]:
# ── AsymmetricLoss (ICCV 2021) ───────────────────────────────────────────────
class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=4, gamma_pos=0, clip=0.05, eps=1e-8):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.eps  = eps

    def forward(self, logits, targets):
        probs     = torch.sigmoid(logits)
        probs_neg = 1.0 - probs
        if self.clip > 0:
            probs_neg = (probs_neg + self.clip).clamp(max=1.0)
        log_p  = torch.log(probs.clamp(min=self.eps))
        log_np = torch.log(probs_neg.clamp(min=self.eps))
        loss   = targets * log_p + (1 - targets) * log_np
        with torch.no_grad():
            w = (targets       * (1 - probs).pow(self.gamma_pos) +
                 (1 - targets) * probs.pow(self.gamma_neg))
        return -(loss * w).mean()

In [ ]:
# ── Dataset ──────────────────────────────────────────────────────────────────
_MEAN = [0.485, 0.456, 0.406]
_STD  = [0.229, 0.224, 0.225]

def get_train_transform(img_size=320):
    return T.Compose([
        T.Resize((img_size, img_size)),
        T.RandomHorizontalFlip(),
        T.RandomRotation(15),
        T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
        T.ToTensor(),
        T.Normalize(_MEAN, _STD),
        T.RandomErasing(p=0.3, scale=(0.02, 0.2)),
    ])

def get_val_transform(img_size=320):
    return T.Compose([
        T.Resize((img_size, img_size)),
        T.ToTensor(),
        T.Normalize(_MEAN, _STD),
    ])

class VOCDataset(Dataset):
    def __init__(self, df, data_dir, split='train', transform=None):
        self.df         = df
        self.data_dir   = Path(data_dir)
        self.split      = split
        self.transform  = transform or get_val_transform()
        self.has_labels = all(c in df.columns for c in LABELS)
        self.indices    = list(df.index)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        arr = np.load(self.data_dir / self.split / 'img' / f'{self.split}_{idx}.npy')
        img = self.transform(Image.fromarray(arr))
        if self.has_labels:
            label = torch.FloatTensor(self.df.loc[idx, LABELS].values.astype(float))
            return img, label
        return img, idx

def load_train_df():
    return pd.read_csv(DATA_DIR / 'train' / 'train_set.csv', index_col='Id')

def load_test_df():
    return pd.read_csv(DATA_DIR / 'test' / 'test_set.csv', index_col='Id')

In [ ]:
# ── Model ────────────────────────────────────────────────────────────────────
class MultiLabelClassifier(nn.Module):
    def __init__(self, backbone='efficientnet_b3', num_classes=20, pretrained=True):
        super().__init__()
        if backbone == 'efficientnet_b3':
            m = models.efficientnet_b3(
                weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1 if pretrained else None
            )
            self.features  = nn.Sequential(m.features, m.avgpool)
            feat_dim = 1536
        elif backbone == 'resnet50':
            m = models.resnet50(
                weights=models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
            )
            self.features = nn.Sequential(*list(m.children())[:-1])
            feat_dim = 2048
        else:
            raise ValueError(f'Unknown backbone: {backbone}')

        self.classifier = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(feat_dim, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.2),
            nn.Linear(512, num_classes),
        )
        for layer in self.classifier.modules():
            if isinstance(layer, nn.Linear):
                nn.init.kaiming_normal_(layer.weight)
                nn.init.zeros_(layer.bias)

    def freeze_backbone(self):
        for p in self.features.parameters(): p.requires_grad = False

    def unfreeze_backbone(self):
        for p in self.features.parameters(): p.requires_grad = True

    def forward(self, x):
        return self.classifier(self.features(x).flatten(1))

In [ ]:
# ── Training helpers ─────────────────────────────────────────────────────────
def run_epoch(model, loader, criterion, optimizer, train):
    model.train(train)
    total_loss = 0.0
    with torch.set_grad_enabled(train):
        for imgs, labels in tqdm(loader, leave=False, desc='train' if train else 'val  '):
            imgs, labels = imgs.to(device), labels.to(device)
            loss = criterion(model(imgs), labels)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(imgs)
    return total_loss / len(loader.dataset)


def train_stage(model, train_loader, val_loader, criterion, lr, epochs, stage_name, best_val_loss):
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    for epoch in range(1, epochs + 1):
        train_loss = run_epoch(model, train_loader, criterion, optimizer, train=True)
        val_loss   = run_epoch(model, val_loader,   criterion, optimizer, train=False)
        scheduler.step()
        flag = ''
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), OUTPUT_DIR / 'best_model.pth')
            flag = '  ← best'
        print(f'[{stage_name}] Epoch {epoch:3d}/{epochs} | '
              f'train={train_loss:.4f}  val={val_loss:.4f}{flag}')
    return best_val_loss

In [ ]:
# ── Run training ─────────────────────────────────────────────────────────────
df = load_train_df()
train_idx, val_idx = train_test_split(range(len(df)), test_size=VAL_SPLIT, random_state=RANDOM_SEED)

train_loader = DataLoader(
    VOCDataset(df.iloc[train_idx], DATA_DIR, split='train', transform=get_train_transform(IMG_SIZE)),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    VOCDataset(df.iloc[val_idx], DATA_DIR, split='train', transform=get_val_transform(IMG_SIZE)),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)

model     = MultiLabelClassifier(backbone=BACKBONE, num_classes=len(LABELS), pretrained=True).to(device)
criterion = AsymmetricLoss(gamma_neg=4, gamma_pos=0, clip=0.05)
best_loss = float('inf')

print(f'\n=== Stage 1: head only (backbone frozen) ===')
model.freeze_backbone()
best_loss = train_stage(model, train_loader, val_loader, criterion,
                        STAGE1_LR, STAGE1_EPOCHS, 'S1', best_loss)

print(f'\n=== Stage 2: full fine-tune ===')
model.unfreeze_backbone()
best_loss = train_stage(model, train_loader, val_loader, criterion,
                        STAGE2_LR, STAGE2_EPOCHS, 'S2', best_loss)

print(f'\n=== Stage 3: retrain on all 750 samples ===')
model.load_state_dict(torch.load(OUTPUT_DIR / 'best_model.pth', map_location=device))
full_loader = DataLoader(
    VOCDataset(df, DATA_DIR, split='train', transform=get_train_transform(IMG_SIZE)),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True
)
opt3  = torch.optim.AdamW(model.parameters(), lr=STAGE3_LR, weight_decay=WEIGHT_DECAY)
sch3  = torch.optim.lr_scheduler.CosineAnnealingLR(opt3, T_max=STAGE3_EPOCHS)
for epoch in range(1, STAGE3_EPOCHS + 1):
    loss = run_epoch(model, full_loader, criterion, opt3, train=True)
    sch3.step()
    print(f'[S3] Epoch {epoch}/{STAGE3_EPOCHS} | loss={loss:.4f}')

torch.save(model.state_dict(), OUTPUT_DIR / 'final_model.pth')
print(f'\nDone. Best val loss: {best_loss:.4f}')
print('Saved: best_model.pth  final_model.pth')

In [ ]:
# ── Threshold optimisation (val set → maximise F1 per class) ─────────────────
model.load_state_dict(torch.load(OUTPUT_DIR / 'best_model.pth', map_location=device))
model.eval()

all_probs, all_labels = [], []
with torch.no_grad():
    for imgs, labels in tqdm(val_loader, desc='Val inference'):
        probs = torch.sigmoid(model(imgs.to(device))).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels.numpy())

all_probs  = np.vstack(all_probs)
all_labels = np.vstack(all_labels)

mAP = average_precision_score(all_labels, all_probs, average='macro')
print(f'Val mAP: {mAP:.4f}')

thresholds = np.zeros(len(LABELS))
for i, cls in enumerate(LABELS):
    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.1, 0.9, 0.05):
        preds = (all_probs[:, i] > t).astype(int)
        tp = (preds * all_labels[:, i]).sum()
        fp = (preds * (1 - all_labels[:, i])).sum()
        fn = ((1 - preds) * all_labels[:, i]).sum()
        f1 = 2 * tp / (2 * tp + fp + fn + 1e-8)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    thresholds[i] = best_t
    print(f'  {cls:<14s}  threshold={best_t:.2f}  F1={best_f1:.3f}')

np.save(OUTPUT_DIR / 'best_thresholds.npy', thresholds)
print('Saved: best_thresholds.npy')

In [ ]:
# ── Generate submission CSV ───────────────────────────────────────────────────
def rle_encode(arr):
    pixels = np.concatenate([[0], arr.flatten(), [0]])
    runs   = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs)


ckpt = OUTPUT_DIR / 'last_model.pth'
model.load_state_dict(torch.load(ckpt, map_location=device))
model.eval()

test_df     = load_test_df()
test_loader = DataLoader(
    VOCDataset(test_df, DATA_DIR, split='test', transform=get_val_transform(IMG_SIZE)),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)

all_probs = []
with torch.no_grad():
    for imgs, _ in tqdm(test_loader, desc='Test inference (TTA)'):
        imgs = imgs.to(device)
        p = (torch.sigmoid(model(imgs)) + torch.sigmoid(model(imgs.flip(-1)))) / 2
        all_probs.append(p.cpu().numpy())

all_probs = np.vstack(all_probs)
preds     = (all_probs > thresholds[None, :]).astype(int)
test_df[LABELS] = preds

rows = {'Id': [], 'Predicted': []}
for idx in test_df.index:
    rows['Id'].append(f'{idx}_classification')
    rows['Predicted'].append(rle_encode(test_df.loc[idx, LABELS].values.astype(int)))
    rows['Id'].append(f'{idx}_segmentation')
    rows['Predicted'].append('')

out = pd.DataFrame(rows).set_index('Id')
out.to_csv(OUTPUT_DIR / 'submission_classification.csv')
print(f'Saved: submission_classification.csv  ({len(out)} rows)')
out.head(4)